# Typing Speed vs Decoding Performance

Reviewer question: *Does the decoding performance depend on typing speed?*

We compute per-subject typing speed (median inter-keystroke interval) from the Button events
and correlate it with Brain2Qwerty character error rate (CER).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import neuralset as ns

DATA_ROOT = os.environ.get("BRAINAI_DATA_ROOT", os.path.expanduser("~/brainai"))
DATA_DIR = f"{DATA_ROOT}/data/typing_decoding"
REVIEWS_CSV = f"{DATA_DIR}/prediction_csv/reviews_MEG_B2Q.csv"

## 1. Load events and compute typing speed

In [ ]:
from pinet_decoding.grids.defaults import default_config

# Load all subjects (remove the single-subject query)
study_cfg = default_config["data"]["study"].copy()
study_cfg.pop("query", None)
loader = ns.data.StudyLoader(**study_cfg)
events = loader.build()

print(f"Raw events: {len(events):,} rows, {events['subject'].nunique()} subjects")

In [ ]:
# Apply the same filtering/merging as preprocessing() but keep readable subject IDs
events = events[~events["trial_id"].isin([0.0, 1.0])]

control_data = [
    "Pinet2024Meg/S11122024", "Pinet2024Meg/S12122024",
    "Pinet2024Meg/S26112024", "Pinet2024Meg/S27112024",
    "Pinet2024Meg/S28112024",
]
events = events[~events["subject"].isin(control_data)]
events = events[events["subject"] != "Pinet2024Meg/S23"]

# Merge same-person sessions
events["subject"] = events["subject"].str.replace("Pinet2024Meg/S18", "Pinet2024Meg/S1")
events["subject"] = events["subject"].str.replace("Pinet2024Meg/S14", "Pinet2024Meg/S4")
events["subject"] = events["subject"].str.replace("Pinet2024Meg/S10", "Pinet2024Meg/S5")
events["subject"] = events["subject"].str.replace("Pinet2024Meg/S21", "Pinet2024Meg/S5")

# Extract integer subject ID to match the reviews CSV
events["subject_id"] = events["subject"].str.extract(r"S(\d+)").astype(int)

# Sentence UID for grouping keystrokes
events["sentence_UID"] = events["trial_id"].astype(str) + "_" + events["timeline"]

buttons = events[events["type"] == "Button"].copy()
print(f"Button events: {len(buttons):,} across {buttons['subject_id'].nunique()} subjects")

In [ ]:
# Compute inter-keystroke interval (IKI) within each sentence
buttons = buttons.sort_values(["sentence_UID", "start"])
buttons["iki"] = buttons.groupby("sentence_UID")["start"].diff()

# Drop the first keystroke of each sentence (no IKI) and outliers (> 5s pauses)
iki = buttons.dropna(subset=["iki"])
iki = iki[iki["iki"] < 5.0]

# Per-subject typing speed statistics
typing_speed = iki.groupby("subject_id")["iki"].agg(
    median_iki="median",
    mean_iki="mean",
    std_iki="std",
    n_keystrokes="count",
).reset_index()

# Convert to keystrokes per second
typing_speed["keys_per_sec"] = 1.0 / typing_speed["median_iki"]

print(typing_speed[["subject_id", "median_iki", "keys_per_sec", "n_keystrokes"]].to_string(index=False))

## 2. Load decoding performance

In [ ]:
reviews = pd.read_csv(REVIEWS_CSV)

# Per-subject mean CER (raw model, and with language model)
perf = reviews.groupby("Subject").agg(
    mean_CER_true=("CER_true", "mean"),
    mean_CER_LM=("CER_LM", "mean"),
    mean_WER_true=("WER_true", "mean"),
    n_sentences=("CER_true", "count"),
).reset_index().rename(columns={"Subject": "subject_id"})

print(perf.to_string(index=False))

## 3. Merge and correlate

In [ ]:
df = typing_speed.merge(perf, on="subject_id")

# Spearman correlation (robust to non-normality)
for metric_col, label in [("mean_CER_true", "CER (raw model)"), ("mean_CER_LM", "CER (with LM)")]:
    r, p = stats.spearmanr(df["median_iki"], df[metric_col])
    print(f"Typing speed (median IKI) vs {label}: Spearman r = {r:.3f}, p = {p:.4f}")

# Also Pearson on keys_per_sec
for metric_col, label in [("mean_CER_true", "CER (raw model)"), ("mean_CER_LM", "CER (with LM)")]:
    r, p = stats.pearsonr(df["keys_per_sec"], df[metric_col])
    print(f"Typing speed (keys/s) vs {label}: Pearson r = {r:.3f}, p = {p:.4f}")

## 4. Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric_col, label in [
    (axes[0], "mean_CER_true", "CER (Conv+Transformer)"),
    (axes[1], "mean_CER_LM", "CER (Conv+Transformer+LM)"),
]:
    ax.scatter(df["keys_per_sec"], df[metric_col], s=60, alpha=0.8, edgecolors="k", linewidths=0.5)

    # Annotate subjects
    for _, row in df.iterrows():
        ax.annotate(f"S{int(row['subject_id'])}", (row["keys_per_sec"], row[metric_col]),
                    fontsize=7, ha="center", va="bottom", xytext=(0, 4), textcoords="offset points")

    # Regression line
    slope, intercept, r_val, p_val, _ = stats.linregress(df["keys_per_sec"], df[metric_col])
    x_range = np.linspace(df["keys_per_sec"].min() - 0.1, df["keys_per_sec"].max() + 0.1, 100)
    ax.plot(x_range, slope * x_range + intercept, "--", color="gray", alpha=0.7)

    r_sp, p_sp = stats.spearmanr(df["keys_per_sec"], df[metric_col])
    ax.set_xlabel("Typing speed (keystrokes / s)", fontsize=12)
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f"Spearman r = {r_sp:.2f}, p = {p_sp:.3f}", fontsize=11)

fig.suptitle("Decoding performance vs typing speed", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("typing_speed_vs_decoding.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSaved: typing_speed_vs_decoding.png")

## 5. Summary table

In [ ]:
summary = df[["subject_id", "keys_per_sec", "median_iki", "mean_CER_true", "mean_CER_LM", "n_keystrokes", "n_sentences"]].copy()
summary = summary.rename(columns={
    "subject_id": "Subject",
    "keys_per_sec": "Typing Speed (keys/s)",
    "median_iki": "Median IKI (s)",
    "mean_CER_true": "CER (raw)",
    "mean_CER_LM": "CER (LM)",
    "n_keystrokes": "# Keystrokes",
    "n_sentences": "# Test Sentences",
}).sort_values("Typing Speed (keys/s)", ascending=False)

print(summary.to_string(index=False))

## 6. Conclusion

Report the Spearman correlation, p-value, and interpretation.
If p > 0.05, we can state that typing speed does **not** significantly predict decoding accuracy.

In [ ]:
r_sp, p_sp = stats.spearmanr(df["keys_per_sec"], df["mean_CER_true"])
r_sp_lm, p_sp_lm = stats.spearmanr(df["keys_per_sec"], df["mean_CER_LM"])

print("="*60)
print("CONCLUSION")
print("="*60)
print(f"N = {len(df)} participants")
print(f"\nTyping speed range: {df['keys_per_sec'].min():.2f} - {df['keys_per_sec'].max():.2f} keys/s")
print(f"  (median IKI range: {df['median_iki'].min()*1000:.0f} - {df['median_iki'].max()*1000:.0f} ms)")
print(f"\nCER (raw model)  vs typing speed: Spearman rho = {r_sp:.3f}, p = {p_sp:.4f}")
print(f"CER (with LM)    vs typing speed: Spearman rho = {r_sp_lm:.3f}, p = {p_sp_lm:.4f}")
print()
if p_sp > 0.05 and p_sp_lm > 0.05:
    print("-> No significant correlation between typing speed and decoding performance.")
    print("   Decoding accuracy does not depend on typing speed.")
else:
    print("-> Significant correlation detected. See plot for direction and effect size.")